# TP-1: Cleaning Data — Applying OOP and Design Patterns



Each exercise builds on the previous one. By the end we have a complete CSV-cleaning pipeline using **OOP, decorators and design patterns**:

| # | Exercise | Concept |
|---|----------|---------|
| 1 | `CSVReader` | Classes, constructors, methods, encapsulation |
| 2 | Missing values | **Strategy** pattern (inheritance, polymorphism) |
| 3 | Logging and timing | **Decorator** pattern (Open/Closed Principle) |
| 4 | Data transformations | **Factory** pattern (abstraction, composition) |
| 5 | Full pipeline | **Template Method** pattern |

Run all cells in order. The notebook expects `sample_data (2).csv` in the same folder.

In [ ]:
from abc import ABC, abstractmethod
from functools import wraps
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

DATA_PATH = Path("sample_data (2).csv")     
OUTPUT_PATH = Path("cleaned_data.csv")      

---
# Exercise 1: Build the `CSVReader` Class (OOP Foundation)

**Objective:** reinforce OOP fundamentals (classes, constructors, methods, encapsulation) and design a reusable data class.

`CSVReader` keeps everything related to reading a CSV file in one place:

- `__init__(file_path)` stores the path (the constructor runs automatically when an object is created).
- `read()` loads the file with `pandas.read_csv()` and keeps the DataFrame in `self.data`.
- `preview(n)` prints the first `n` rows.

In [2]:
class CSVReader:
    """Encapsulates reading and previewing a CSV file."""

    def __init__(self, file_path):
        self.file_path = Path(file_path)
        self.data = None

    def read(self):
        """Load the CSV file into a DataFrame (stored in self.data)."""
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        """Print the first n rows (reads the file first if needed)."""
        if self.data is None:
            self.read()
        print(self.data.head(n))

In [ ]:
reader = CSVReader(DATA_PATH)
df = reader.read()
print(f"Loaded {DATA_PATH}: {df.shape[0]} rows x {df.shape[1]} columns\n")
reader.preview(5)
print()
reader.preview(3)    

Loaded sample_data (2).csv: 7 rows x 7 columns

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN


---
# Exercise 2: Strategy Pattern for Missing Values

**Objective:** one interface, many interchangeable behaviours; practise inheritance and polymorphism.

- `MissingValueStrategy` is the abstract interface with `handle(df)`.
- `DropMissing`, `FillMean`, `FillMode` are the concrete strategies.
- `DataCleaner` applies whichever strategy it is given, so the strategy can change **without changing the main code**.

Every strategy returns a new DataFrame and leaves the input untouched.

In [4]:
class MissingValueStrategy(ABC):
    """Interface shared by all missing-value strategies."""

    @abstractmethod
    def handle(self, df):
        """Return a cleaned copy of df."""


class DropMissing(MissingValueStrategy):
    """Remove every row that contains a missing value."""

    def handle(self, df):
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    """Fill missing numeric values with the column mean."""

    def handle(self, df):
        result = df.copy()
        numeric_cols = result.select_dtypes(include="number").columns
        result[numeric_cols] = result[numeric_cols].fillna(result[numeric_cols].mean())
        return result


class FillMode(MissingValueStrategy):
    """Fill missing values with the most frequent value of each column."""

    def handle(self, df):
        result = df.copy()
        for column in result.columns:
            modes = result[column].mode()
            if result[column].isna().any() and not modes.empty:
                result[column] = result[column].fillna(modes.iloc[0])
        return result


class DataCleaner:
    """Context class: delegates missing-value handling to a strategy."""

    def __init__(self, strategy):
        self.strategy = strategy

    def set_strategy(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

In [ ]:
raw_df = CSVReader(DATA_PATH).read()
print("Original data (NaN = missing):")
display(raw_df)

cleaner = DataCleaner(DropMissing())
for strategy in (DropMissing(), FillMean(), FillMode()):
    cleaner.set_strategy(strategy)              
    result = cleaner.clean(raw_df)
    print(f"\n{type(strategy).__name__}  ->  {result.isna().sum().sum()} missing cells left")
    display(result)

Original data (NaN = missing):


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0



DropMissing  ->  0 missing cells left


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,5,Eva,27.0,160.0,55.0,New York,88.0



FillMean  ->  0 missing cells left


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0



FillMode  ->  0 missing cells left


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


---
# Exercise 3: Decorators for Logging and Timing

**Objective:** add reusable behaviour (logging, timing) and see how decorators support the **Open/Closed Principle**: classes stay closed for modification but open for extension.

- `@log_action` logs when a method **starts** and **finishes** (and still reports a failure).
- `@log_time` measures and prints the **execution time**.

`functools.wraps` keeps the original name and docstring of the wrapped function.

In [6]:
def log_action(func):
    """Log when a function/method starts and finishes."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG]  Starting {func.__qualname__}")
        try:
            return func(*args, **kwargs)
        except Exception as exc:
            print(f"[LOG]  {func.__qualname__} FAILED: {exc}")
            raise
        finally:
            print(f"[LOG]  Finished {func.__qualname__}")
    return wrapper


def log_time(func):
    """Measure and print the execution time of a function/method."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"[TIME] {func.__qualname__} took {perf_counter() - start:.6f}s")
    return wrapper

### Applying the decorators to `CSVReader` and `DataCleaner`

The class bodies from Exercises 1 and 2 are **not edited**. We extend them from the outside by wrapping `CSVReader.read` and `DataCleaner.clean`. With `log_action` outermost, START prints first, FINISH last, and the timer measures only the method itself.

In [7]:
CSVReader.read = log_action(log_time(CSVReader.read))
DataCleaner.clean = log_action(log_time(DataCleaner.clean))

# Metadata is preserved thanks to functools.wraps
print(CSVReader.read.__name__, "|", DataCleaner.clean.__name__)

read | clean


In [8]:
reader = CSVReader(DATA_PATH)
df = reader.read()
print()
cleaned = DataCleaner(FillMean()).clean(df)
display(cleaned)

[LOG]  Starting CSVReader.read
[TIME] CSVReader.read took 0.006868s
[LOG]  Finished CSVReader.read

[LOG]  Starting DataCleaner.clean
[TIME] DataCleaner.clean took 0.001990s
[LOG]  Finished DataCleaner.clean


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


---
# Exercise 4: Factory Pattern for Data Transformations

**Objective:** create transformation objects through a factory (scalable creation), applying abstraction and composition.

- `DataTransform` is the abstract interface with `apply(df)`.
- `NormalizeColumns` scales numeric columns to the range [0, 1] (min-max). Identifier columns (default: `id`) are left unchanged.
- `RemoveDuplicates` drops identical rows.
- `StandardizeText` strips whitespace and lowercases text values.
- `TransformFactory.create(name)` returns the right object from a string, and raises a clear error for unknown names.

In [9]:
class DataTransform(ABC):
    """Interface shared by all transformations."""

    @abstractmethod
    def apply(self, df):
        """Return a transformed copy of df."""


class NormalizeColumns(DataTransform):
    """Min-max scale numeric columns to [0, 1]; identifier columns are skipped."""

    def __init__(self, exclude=("id",)):
        self.exclude = set(exclude)

    def apply(self, df):
        result = df.copy()
        for column in result.select_dtypes(include="number").columns:
            if column in self.exclude:
                continue
            low, high = result[column].min(), result[column].max()
            result[column] = 0.0 if high == low else (result[column] - low) / (high - low)
        return result


class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    def apply(self, df):
        result = df.copy()
        for column in result.select_dtypes(include=["object", "string"]).columns:
            result[column] = result[column].map(
                lambda v: v.strip().lower() if isinstance(v, str) else v
            )
        return result


class TransformFactory:
    """Creates transformation objects from string names."""

    _registry = {
        "normalize": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create(cls, name):
        key = name.strip().lower()
        if key not in cls._registry:
            available = ", ".join(sorted(cls._registry))
            raise ValueError(f"Unknown transformation {name!r}. Available: {available}")
        return cls._registry[key]()

In [10]:
# Input: the fully cleaned data from Exercise 3 (FillMean strategy)
for name in ("standardize_text", "normalize", "remove_duplicates"):
    out = TransformFactory.create(name).apply(cleaned)
    print(f"{name} ->  {type(TransformFactory.create(name)).__name__}")
    display(out)

# Composition: chain several transformations created by the factory
result = cleaned.copy()
for name in ("standardize_text", "normalize", "remove_duplicates"):
    result = TransformFactory.create(name).apply(result)
print("Chained result:")
display(result)

# Unknown names fail with a helpful message
try:
    TransformFactory.create("unknown")
except ValueError as error:
    print("Error:", error)

standardize_text ->  StandardizeText


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.00,165.0,68.0,new york,85.0
1,2,bob,30.25,172.0,70.0,los angeles,90.0
2,3,charlie,35.00,168.0,72.0,chicago,86.0
3,4,david,30.25,167.0,80.0,houston,75.0
4,5,eva,27.00,160.0,55.0,new york,88.0
5,6,frank,30.00,175.0,85.0,los angeles,86.0
6,7,grace,30.25,162.0,60.0,chicago,92.0


normalize ->  NormalizeColumns


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,2,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,3,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,4,David,0.40625,0.466667,0.833333,Houston,0.000000
4,5,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,6,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,7,Grace,0.40625,0.133333,0.166667,Chicago,1.000000


remove_duplicates ->  RemoveDuplicates


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


Chained result:


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,0.25000,0.333333,0.433333,new york,0.588235
1,2,bob,0.40625,0.800000,0.500000,los angeles,0.882353
2,3,charlie,1.00000,0.533333,0.566667,chicago,0.647059
3,4,david,0.40625,0.466667,0.833333,houston,0.000000
4,5,eva,0.00000,0.000000,0.000000,new york,0.764706
5,6,frank,0.37500,1.000000,1.000000,los angeles,0.647059
6,7,grace,0.40625,0.133333,0.166667,chicago,1.000000


Error: Unknown transformation 'unknown'. Available: normalize, remove_duplicates, standardize_text


---
# Exercise 5: Full Cleaning Pipeline (Template Method Pattern)

**Objective:** combine everything into one pipeline with a consistent workflow.

- `DataPipeline.run()` is the **template**: it fixes the order `load() -> clean() -> transform() -> save()`.
- `CSVDataPipeline` implements each step using the earlier classes:

| Step | Uses |
|------|------|
| `load()` | `CSVReader` (Ex. 1) |
| `clean()` | `DataCleaner` + a `MissingValueStrategy` (Ex. 2) |
| `transform()` | `TransformFactory` (Ex. 4) |
| `save()` | `DataFrame.to_csv` |

`run()` itself is wrapped with `@log_action` and `@log_time` (Ex. 3).

In [11]:
class DataPipeline(ABC):
    """Template: defines the skeleton of the workflow; subclasses fill in the steps."""

    @log_action
    @log_time
    def run(self):
        df = self.load()
        df = self.clean(df)
        df = self.transform(df)
        self.save(df)
        return df

    @abstractmethod
    def load(self):
        ...

    @abstractmethod
    def clean(self, df):
        ...

    @abstractmethod
    def transform(self, df):
        ...

    @abstractmethod
    def save(self, df):
        ...


class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, strategy, transforms=()):
        self.reader = CSVReader(input_path)
        self.cleaner = DataCleaner(strategy)
        self.transforms = [TransformFactory.create(name) for name in transforms]
        self.output_path = Path(output_path)

    def load(self):
        return self.reader.read()

    def clean(self, df):
        return self.cleaner.clean(df)

    def transform(self, df):
        for transform in self.transforms:
            df = transform.apply(df)
        return df

    def save(self, df):
        df.to_csv(self.output_path, index=False)
        print(f"Saved cleaned data to: {self.output_path}")

In [12]:
pipeline = CSVDataPipeline(
    input_path=DATA_PATH,
    output_path=OUTPUT_PATH,
    strategy=FillMean(),
    transforms=["standardize_text", "normalize", "remove_duplicates"],
)
final_df = pipeline.run()

print("\nBefore cleaning:")
display(pd.read_csv(DATA_PATH))
print("After cleaning:")
display(final_df)

[LOG]  Starting DataPipeline.run
[LOG]  Starting CSVReader.read
[TIME] CSVReader.read took 0.001099s
[LOG]  Finished CSVReader.read
[LOG]  Starting DataCleaner.clean
[TIME] DataCleaner.clean took 0.001330s
[LOG]  Finished DataCleaner.clean
Saved cleaned data to: cleaned_data.csv
[TIME] DataPipeline.run took 0.006206s
[LOG]  Finished DataPipeline.run

Before cleaning:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


After cleaning:


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,0.25000,0.333333,0.433333,new york,0.588235
1,2,bob,0.40625,0.800000,0.500000,los angeles,0.882353
2,3,charlie,1.00000,0.533333,0.566667,chicago,0.647059
3,4,david,0.40625,0.466667,0.833333,houston,0.000000
4,5,eva,0.00000,0.000000,0.000000,new york,0.764706
5,6,frank,0.37500,1.000000,1.000000,los angeles,0.647059
6,7,grace,0.40625,0.133333,0.166667,chicago,1.000000


In [13]:
# Sanity checks
saved = pd.read_csv(OUTPUT_PATH)
assert final_df.isna().sum().sum() == 0, "missing values remain"
assert saved.shape == final_df.shape and list(saved.columns) == list(final_df.columns)
assert saved.select_dtypes("number").drop(columns="id").stack().between(0, 1).all()
print("Pipeline checks passed.")

Pipeline checks passed.


---
## Summary

| Exercise | Pattern / concept | Classes |
|----------|-------------------|---------|
| 1 | OOP foundation | `CSVReader` |
| 2 | Strategy | `MissingValueStrategy`, `DropMissing`, `FillMean`, `FillMode`, `DataCleaner` |
| 3 | Decorator | `log_action`, `log_time` |
| 4 | Factory | `DataTransform`, `NormalizeColumns`, `RemoveDuplicates`, `StandardizeText`, `TransformFactory` |
| 5 | Template Method | `DataPipeline`, `CSVDataPipeline` |

Adding a new missing-value strategy or transformation only requires a new subclass (plus one registry entry for the factory); the pipeline code stays unchanged.